## **Setup**

In [1]:

%pip install -q --progress-bar off --no-warn-conflicts "langchain>=0.2" langchain-community langchain-core langchain-ollama langchain-docling python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_docling.loader import ExportType

load_dotenv()

# https://github.com/huggingface/transformers/issues/5486:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

#FILE_PATH = ["https://arxiv.org/pdf/2408.09869"]  # Docling Technical Report
#EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
#GEN_MODEL_ID = "mistralai/Mixtral-8x7B-Instruct-v0.1"
EXPORT_TYPE = ExportType.DOC_CHUNKS
QUESTION = "Please summarise this document?"
PROMPT = PromptTemplate.from_template(
    "Context information is below.\n---------------------\n{context}\n---------------------\nGiven the context information and not prior knowledge, answer the query.\nQuery: {input}\nAnswer:\n",
)
TOP_K = 3

C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **Document Loading**

In [3]:
# Custom Settings
FILE_PATH = "my_papers/2024 - 15.pdf"
EXPORT_TYPE = ExportType.DOC_CHUNKS
EMBED_MODEL_ID = "mxbai-embed-large:latest"
GEN_MODEL_ID = "llama3.1:8b-instruct-q4_K_M"
# Tokenizer for chunking must be a HF repo id, not an Ollama tag
CHUNK_TOKENIZER_ID = "sentence-transformers/all-MiniLM-L6-v2"

In [4]:
from langchain_docling import DoclingLoader

from docling.chunking import HybridChunker

loader = DoclingLoader(
    file_path=FILE_PATH,
    export_type=EXPORT_TYPE,
    chunker=HybridChunker(tokenizer=CHUNK_TOKENIZER_ID),
)

docs = loader.load()

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-02-10 06:51:42,644 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-10 06:51:42,669 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-02-10 06:51:42,714 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-10 06:51:42,716 [RapidOCR] main.py:50: Using C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-10 06:51:42,953 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-10 06:51:42,954 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-02-10 06:51:42,960 [RapidOCR] download_file.py:60: File exists and is valid: C:

*Ignore  this warning*: "Token indices sequence length is longer than the specified maximum sequence length for this model (638 > 512)."

In [5]:
if EXPORT_TYPE == ExportType.DOC_CHUNKS:
    splits = docs
elif EXPORT_TYPE == ExportType.MARKDOWN:
    from langchain_text_splitters import MarkdownHeaderTextSplitter

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "Header_1"),
            ("##", "Header_2"),
            ("###", "Header_3"),
        ],
    )
    splits = [split for doc in docs for split in splitter.split_text(doc.page_content)]
else:
    raise ValueError(f"Unexpected export type: {EXPORT_TYPE}")

Inspecting some splits

In [6]:
for d in splits[:3]:
    print(f"- {d.page_content=}")
print("...")

- d.page_content='JHTT 15,4\n592\nReceived22 June 2023 Revised 11 September 2023 21 November 2023 7 February 2024 Accepted16 April 2024\nJournal of Hospitality and Tourism Technology Vol. 15 No. 4, 2024 pp. 592-609\n©EmeraldPublishingLimited 1757-9880 DOI 10.1108/JHTT-06-2023-0170'
- d.page_content="Anovel ChatGPT-based multimodel framework for tourism review mining: a case study on China ' s /uniFB01 ve sacred mountains\nXinquan Cheng\nDepartment of Tourism Management, Pai Chai University, Daejeon, Republic of Korea"
- d.page_content='Yuanhong Chen\nSchool of Creativity and Management, Communication University of Shanxi, Jinzhong, China and Department of Tourism Management, Pai Chai University, Daejeon, Republic of Korea'
...


## **Ingestion**

In [7]:
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_core.vectorstores import InMemoryVectorStore

In [8]:
#embedding = HuggingFaceEmbeddings(model_name=EMBED_MODEL_ID)
embedding = OllamaEmbeddings(model=EMBED_MODEL_ID)

# In-memory vector store (no external DB)
vectorstore = InMemoryVectorStore.from_documents(
    documents=splits,
    embedding=embedding,
)

## **RAG**

In [12]:
import json


from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
llm = OllamaLLM(model=GEN_MODEL_ID)

def clip_text(text, threshold=100):
    return f"{text[:threshold]}..." if len(text) > threshold else text

In [13]:
question_answer_chain = create_stuff_documents_chain(llm, PROMPT)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
resp_dict = rag_chain.invoke({"input": QUESTION})

clipped_answer = clip_text(resp_dict["answer"], threshold=200)
print(f"Question:\n{resp_dict['input']}\n\nAnswer:\n{clipped_answer}")
for i, doc in enumerate(resp_dict["context"]):
    print()
    print(f"Source {i + 1}:")
    print(f"  text: {json.dumps(clip_text(doc.page_content, threshold=350))}")
    for key in doc.metadata:
        if key != "pk":
            val = doc.metadata.get(key)
            clipped_val = clip_text(val) if isinstance(val, str) else val
            print(f"  {key}: {clipped_val}")

Question:
Please summarise this document?

Answer:
This document appears to be a research paper that discusses a methodology for analyzing customer reviews. The paper seems to focus on using ChatGPT-assessed sentiment scores to analyze the importance ...

Source 1:
  text: "3. Research methods\nThe overall research framework of the paper is shown in Figure 2."
  source: my_papers/2024 - 15.pdf
  dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/71', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 5, 'bbox': {'l': 99.78, 't': 479.4980024414062, 'r': 345.705, 'b': 470.9380024414062, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 65]}]}], 'headings': ['3. Research methods'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 18434708872324074617, 'filename': '2024 - 15.pdf'}}

Source 2:
  text: "JHTT 15,4\n598\n- 3.Returnthepreparedtext."
  source: m